## Cross checking all pois collected for swimming pools to see those missing at differnt  times.

 If not done already, please share a detailed list of problematic places with Hewan. Be very clear about which place ID was not found at which place. E.g. some places were not found during the Germany-wide collection but, later, were found during the search for the specific place. Also be sure to give examples for both "temporarily closed" and cases where it is not "temporarily closed". For these examples, include specific Google Maps URLs to facilitate things for Hewan. Include at least a few different examples in each category. (By category I mean stages at which a places was [not] found.) -- Of course, also give him the full list of all Google Maps swimming pool POIs that you've discovered through various means as this will also be a helpful check for him. (At the very least we should be able to re-find those, even though that list itself is also not complete.) Thanks!
 

In [ ]:
import pandas as pd
import geopandas as gpd

### swimming pools for state capitals

In [ ]:
list_place_types = ['aquatic center', 'indoor swimming pool', 'lake', 'lake shore swimming area','lido',
                    'outdoor bath', 'outdoor swimming pool', 'public bath', 'public bathroom', 
                    'public sauna', 'public beach','public swimming pool', 'recreation center', 'spa','spa and health club',
                    'sports complex', 'swimming facility', 'swimming lake','swimming pool', 'thermal baths',
                    'water', 'water park']

In [ ]:
len(list_place_types)

In [ ]:
sc_pools = pd.read_csv('/home/ethel/Documents/RESEARCH/swimming_pools_data_16gsc/annotations_12092025/relevant_gsc_pools.csv')
sc_pools

In [ ]:
sc_pools.primary_place_type.value_counts()

In [ ]:
## pids initially collected for all of germany
pids_initial = gpd.read_file('/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/public_swimmingpools_germany/clean_datasets/pois_within_shapefile.gpkg')

pids_initial.info()

In [ ]:
pids_initial.primary_place_type.value_counts()

In [ ]:
#reverse geocode pool locations to get spatial data
pids_geocoded = gpd.read_file('/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/GERMAN_PUBLIC_POOLS/clean_datasets/pois_within_shapefile.gpkg')
#pids_geocoded =  pids_geocoded[pids_geocoded['place_id'].isin(pids_geo_latlong['place_id'])].reset_index(drop=True)
pids_geocoded.info()

In [ ]:
pids_geocoded_2 = pd.read_csv("/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/GERMAN_PUBLIC_POOLS/rev_419/pois_419.csv")
pids_geocoded_2.info()

In [ ]:
pids_geocoded = pd.concat([pids_geocoded, pids_geocoded_2], ignore_index=True)
pids_geocoded.info()

In [ ]:
pids_geocoded.primary_place_type.value_counts()

In [ ]:
#make the primary place type lower case
df1=sc_pools[sc_pools['primary_place_type'].str.lower().isin(list_place_types)]
df2 = pids_initial[pids_initial['primary_place_type'].str.lower().isin(list_place_types)]
df3 = pids_geocoded[pids_geocoded['primary_place_type'].str.lower().isin(list_place_types)]

In [ ]:
print(df1.shape)
print(df2.shape)
print(df3.shape)

In [ ]:
key = "place_id"   # <-- change to your unique identifier

df1_not_in_df2 = df1[~df1[key].isin(df2[key])]
df1_in_df2     = df1[df1[key].isin(df2[key])]

In [ ]:
df1_not_in_df2

In [ ]:
df1_in_df2 

In [ ]:
df3_not_in_df2 = df3[~df3[key].isin(df2[key])]
df3_in_df2     = df3[df3[key].isin(df2[key])]


In [ ]:
df3_not_in_df2 

In [ ]:
df3_in_df2 

In [ ]:
df1_not_in_df3 = df1[~df1[key].isin(df3[key])]
df1_in_df3  = df1[df1[key].isin(df3[key])]

In [ ]:
df1_not_in_df3

In [ ]:
df1_in_df3 

In [ ]:
ids_df1 = set(df1[key])
ids_df2 = set(df2[key])
ids_df3 = set(df3[key])

In [ ]:
all_ids = ids_df1 | ids_df2 | ids_df3   # union


In [ ]:
merged_ids = pd.concat([df1, df2, df3])
merged_ids =merged_ids[['place_id', 'place_name', 'primary_place_type', 'place_subtype', 'google_url', 'website']]

In [ ]:
out = pd.DataFrame({key: list(all_ids)})

out["state_capital_search"] = out[key].apply(lambda x: "yes" if x in ids_df1 else "no")
out["german_wide_search"] = out[key].apply(lambda x: "yes" if x in ids_df2 else "no")
out["geocoded_search"] = out[key].apply(lambda x: "yes" if x in ids_df3 else "no")

In [ ]:
check_list = pd.merge(out, merged_ids, on='place_id', how='left')
check_list = check_list.drop_duplicates()
check_list.info()

In [ ]:
final_pools = pd.read_csv('/home/ethel/Desktop/WEBCONF_25/datasets/final_swimmingpool_list_corrected_23112025.csv')
final_pools.info()

In [ ]:
ccheck_list_final = check_list[check_list['place_id'].isin(final_pools['place_id'])]
ccheck_list_final = ccheck_list_final.drop_duplicates(subset=['place_id']).reset_index(drop=True)
ccheck_list_final.info()

In [ ]:
ccheck_list_final.to_csv('/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/TheWEBConf_TheWeb4Good_2026/data/pois_with_issues_checklist_23112025.csv', index=False)

## check the percentage coverage of gmaps pids compared to the admin data 

In [ ]:
final_list = pd.read_csv("/home/ethel/Desktop/WEBCONF_25/datasets/final_swimmingpool_list_corrected_23112025.csv")
final_list.info()


In [ ]:
admin_pools = pd.read_csv('/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/GERMAN_PUBLIC_POOLS/groundtruth_pools/baederleben_pools_16112025.csv')
admin_pools.info()

In [ ]:
matched_pools = pd.read_csv("/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/GERMAN_PUBLIC_POOLS/matching_coordinates/complete_matched_pairs_all_columns.csv")
matched_pools.info()